# 06 · A truss: joint equilibrium as a linear system

**Why this is needed:** a graph stores the structural model but does not balance forces; an equilibrium matrix connects that model to a numerical solver. This is an optional extension after the matrix lesson.

**CAS Computational Form-finding · Block Research Group · ETH Zürich**

A triangular, pin-jointed truss carries one downward apex load. The COMPAS graph
stores geometry, connectivity and results; NumPy and SciPy solve the force equations.
Members carry axial forces, joints transfer no moments, and loads act at joints.
This is a **static force calculation on given geometry**, not a deformation solver.

Use metres and kN. Work in the world **x–z plane**: +x is right, +z is up.
A member force is **positive in tension**, negative in compression. Reactions are
positive along +x or +z. **Predict:** will the two sloping members be in compression?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
import numpy as np
from scipy.linalg import qr, solve_triangular
from compas.datastructures import Graph
from compas.geometry import Pointcloud
from compas_notebook import Viewer
from compas_notebook.config import Config


## 2. Store the structural model in a graph
Node `left` is a pin: its horizontal and vertical reactions are unknown.
Node `right` is a roller: only its vertical reaction is unknown.
Node `top` carries the downward load. The model therefore has three member
forces and three reactions, balanced by six joint-equilibrium equations.


In [ ]:
span = 4.0  # m
rise = 2.0  # m
P = 10.0   # downward apex load [kN]
assert span > 0 and rise > 0 and P >= 0
truss = Graph()
for key, xyz in {"left": [0, 0, 0], "top": [span / 2, 0, rise],
                 "right": [span, 0, 0]}.items():
    truss.add_node(key, x=xyz[0], y=xyz[1], z=xyz[2], px=0.0, pz=0.0)
truss.node_attribute("top", "pz", -P)
# Keep this list: it defines member-force column order.
members = [("left", "top"), ("top", "right"), ("left", "right")]
for edge in members:
    truss.add_edge(*edge)
reaction_dofs = [("left", "x"), ("left", "z"), ("right", "z")]
truss.attributes.update({"length_unit": "m", "force_unit": "kN", "force_sign": "tension positive"})
print(truss)


## 3. Assemble the joint-equilibrium matrix explicitly
Rows are `(left x, left z, top x, top z, right x, right z)` in node iteration order.
Columns are the three member forces followed by `(left Rx, left Rz, right Rz)`.
For an edge from u to v, a positive tension pulls u **towards v** and v **towards u**.
Put the unit direction in u's rows and its negative in v's rows.

The complete equations are `E @ unknowns + loads = 0`, or `E @ unknowns = -loads`.
Reaction columns contain a single +1 in the corresponding joint equation.


In [ ]:
node_keys = list(truss.nodes())
key_to_row = truss.node_index()
E = np.zeros((2 * len(node_keys), len(members) + len(reaction_dofs)))
for column, (u, v) in enumerate(members):
    delta = np.array(truss.node_coordinates(v)) - np.array(truss.node_coordinates(u))
    direction_xz = delta[[0, 2]] / np.linalg.norm(delta)
    E[2 * key_to_row[u]:2 * key_to_row[u] + 2, column] = direction_xz
    E[2 * key_to_row[v]:2 * key_to_row[v] + 2, column] = -direction_xz
for j, (key, axis) in enumerate(reaction_dofs, start=len(members)):
    row = 2 * key_to_row[key] + {"x": 0, "z": 1}[axis]
    E[row, j] = 1.0
loads = np.array([truss.node_attributes(key, ["px", "pz"]) for key in node_keys]).ravel()
print("Row map:", {key: (2 * row, 2 * row + 1) for key, row in key_to_row.items()})
print("Equilibrium matrix E:\n", np.round(E, 3))
assert E.shape == (6, 6) and np.linalg.matrix_rank(E) == 6


## 4. Solve using QR factorisation
QR expresses `E = Q @ R`, where Q is orthogonal and R is upper triangular.
Multiply by `Q.T`, then solve `R @ unknowns = Q.T @ (-loads)`.
A rank-deficient matrix would indicate that this equilibrium solve needs a
review of geometry, members or supports. The matrix is square and full rank here.


In [ ]:
Q, R = qr(E, mode="economic")
unknowns = solve_triangular(R, Q.T @ (-loads))
forces = unknowns[:len(members)]
reactions = unknowns[len(members):]
for edge, force in zip(members, forces):
    truss.edge_attribute(edge, "force", float(force))
    state = "tension" if force > 0 else "compression" if force < 0 else "zero"
    print(f"{edge}: {force:+.6f} kN ({state})")
for (key, axis), force in zip(reaction_dofs, reactions):
    truss.node_attribute(key, "r" + axis, float(force))
    print(f"Reaction {key} {axis}: {force:+.6f} kN")


## 5. Check joint balance and a separate hand solution
By symmetry each vertical reaction is `P/2`. Each diagonal carries compression
`−P L/(2 rise)`, where L is its length, and the bottom tie carries tension
`P span/(4 rise)`. These formulas remain useful when you edit span, rise or P.

Starting result: sloping forces **−7.071068 kN** each, tie **+5 kN**;
horizontal reaction **0 kN**, vertical reactions **+5 kN** each.


In [ ]:
residual = E @ unknowns + loads
half_span = span / 2
length = np.hypot(half_span, rise)
expected_forces = [-P * length / (2 * rise), -P * length / (2 * rise), P * span / (4 * rise)]
assert np.allclose(forces, expected_forces, atol=1e-9)
assert np.allclose(reactions, [0, P / 2, P / 2], atol=1e-9)
assert np.max(np.abs(residual)) < 1e-9
assert np.isclose(reactions[1] + reactions[2] - P, 0)
assert np.isclose(reactions[2] * span - P * half_span, 0)  # moment about left
print(f"Largest joint residual: {np.max(np.abs(residual)):.2e} kN")

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [5, -8, 4]
config.view.camera.target = [span / 2, 0, rise / 2]
view = Viewer(config=config)
view.scene.add(truss, color="#5552bb", nodesize=0.10)
view.scene.add(Pointcloud([truss.node_coordinates(k) for k in ["left", "right"]]),
               color="#348f9e", pointsize=0.20)
view.show()


## 6. Predict → change → check
1. Double P. Predict every member force and reaction before rerunning.
2. Increase the rise while keeping the span fixed. What happens to the tie force?
3. Point to the entries in E that balance the top joint's vertical load.
4. Explain why this calculation cannot report the displacement of the top joint.

Changing connectivity or supports changes the unknowns and may require a different
matrix size; begin by editing span, rise and P only.


In [ ]:
# Safe experiment: linear scaling of load on the same geometry.
trial_unknowns = solve_triangular(R, Q.T @ (-2 * loads))
assert np.allclose(trial_unknowns, 2 * unknowns)
print("Twice the load produces twice every member force and reaction.")


## Sources
Original course equilibrium derivation. Data model:
[COMPAS Graph](https://compas.dev/compas/2.15.1/api/compas.datastructures.html); geometry/topology foundation:
[ITA21](https://github.com/compas-teaching/ITA21/tree/7213b322067efe6102e998e83194e8dbe39a82bf). Numerical algorithm:
[SciPy QR](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.qr.html)
and [triangular solve](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_triangular.html).
Viewer pattern: [COMPAS JupyterLite](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
